# Chapter 10: Agent Evaluation Frameworks and Benchmarking
## Metrics, Judges, and Production Evaluation - Code Examples

This notebook contains the runnable code from Chapter 10. It builds a small tool-using agent to evaluate, then applies every metric and evaluation technique the chapter describes to that agent's real runs:
- Task completion, tool selection accuracy, and cost efficiency metrics computed from live agent runs
- The chapter's `CostTracker`, extended with current list prices, and a model cascading comparison
- LangSmith tracing and dataset creation, run when a LangSmith key is present and reported as skipped otherwise, plus a trace read directly from the agent's messages
- LLM-as-a-judge scoring with a structured output, calibrated against hand-written human scores
- An evaluation report with regression gates, the shape of an offline evaluation in a CI pipeline

### Setup

The dependencies for every chapter are declared in `pyproject.toml` at the repository root. From the root, run:

```bash
uv sync --all-groups
```

Then start Jupyter with `uv run jupyter lab` and select the **Agentic AI Handbook (Python 3.13)** kernel.

This notebook calls the OpenAI API. Copy `.env.example` to `.env` at the repository root and add your `OPENAI_API_KEY` before running the cells. `LANGSMITH_API_KEY` is optional: with it, the two LangSmith cells trace the agent and create a dataset in your workspace; without it, they print that they were skipped and the rest of the notebook is unaffected.

### How the evaluation runs

The chapter's listings are metrics and evaluators, and they need an agent to measure. Part 1 builds one: a support agent on `gpt-5.6-luna` with three tools (documentation search, order lookup, arithmetic) and a five-task evaluation set with expected tool sequences and expected answers. Every later listing consumes the results of those runs, so the numbers you see are measured, not illustrative. The benchmark section of the chapter (AgentBench, GAIA, SWE-bench) has no listings, because those benchmarks ship their own harnesses; Part 5 points at them.

The chapter's `CostTracker` lists GPT-5 and GPT-5 nano prices. The notebook adds the `gpt-5.6-luna` and `gpt-5.6-terra` rates from OpenAI's pricing page in September 2026, since those are the models it calls. Every other listing is reproduced as printed.

In [ ]:
# Import required libraries
import os
import json
import time
import tempfile
from pathlib import Path
from typing import Any, Dict, List, Optional
from dotenv import load_dotenv

# Load environment variables (API keys)
load_dotenv()

# Verify API keys are loaded
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found in environment"
HAS_LANGSMITH = bool(os.getenv("LANGSMITH_API_KEY"))

# Tracing stays off unless Part 6 turns it on with a key
os.environ.setdefault("LANGSMITH_TRACING", "false")

MODEL = "gpt-5.6-luna"

# Every file this notebook writes lives in a throwaway workspace
WORKSPACE = Path(tempfile.mkdtemp(prefix="ch10-eval-")).resolve()
os.chdir(WORKSPACE)

print("Environment setup complete")
print(f"Workspace: {WORKSPACE}")
print(f"LangSmith key present: {HAS_LANGSMITH}")

## Part 1: The Agent Under Evaluation

Evaluation needs a subject. The agent below answers support questions with three tools: a documentation search over three canned passages, an order lookup over two orders, and an arithmetic calculator. The evaluation set has five tasks, each with the tool sequence a competent agent should use and a short list of phrasings, any one of which marks the answer correct. Exact matching on free text is brittle, which is why the chapter reaches for a judge in Part 7; the phrasing list is the cheap check that runs first. Task 4 needs all three tools in order, which is where tool selection and cost metrics become interesting.

`run_task` records what the metrics need: the final answer, the tools called in order, the token counts from every model call (LangChain exposes them as `usage_metadata` on each AI message), the number of model calls, and wall-clock latency.

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

DOCS = {
    "refund policy": "Refunds are accepted within 30 days of delivery. Items must be unused.",
    "escalation": "Escalate to a supervisor when a customer disputes a charge twice or requests a manager.",
    "shipping": "Standard shipping takes 3 to 5 business days. Express shipping takes 1 to 2 days.",
}
ORDERS = {
    "A100": {"status": "shipped", "days_since_delivery": None},
    "B200": {"status": "delivered", "days_since_delivery": 12},
}


@tool
def search_docs(query: str) -> str:
    """Search the support documentation and return the most relevant passage."""
    q = query.lower()
    for key, text in DOCS.items():
        # Match on word stems so "escalate" finds the "escalation" passage
        if any(word[:4] in q for word in key.split()):
            return text
    return "No matching documentation."


@tool
def lookup_order(order_id: str) -> str:
    """Look up an order by id and return its status as JSON."""
    return json.dumps(ORDERS.get(order_id.upper(), {"error": "unknown order"}))


@tool
def calculate(expression: str) -> str:
    """Evaluate an arithmetic expression such as 30 - 12 and return the result."""
    allowed = set("0123456789+-*/(). ")
    if set(expression) - allowed:
        return "error: only arithmetic is allowed"
    return str(eval(expression))  # arithmetic only; the character check above guards it


TOOLS = [search_docs, lookup_order, calculate]
SYSTEM_PROMPT = "You are a support agent. Use the tools for facts and arithmetic. Answer in one or two sentences."


def build_agent(model: str = MODEL):
    return create_agent(ChatOpenAI(model=model, reasoning_effort="none"), TOOLS, system_prompt=SYSTEM_PROMPT)


TASKS = [
    {"task_id": "1", "input": "What is 17 times 23?",
     "expected_tools": ["calculate"], "expected_answer": ["391"]},
    {"task_id": "2", "input": "What is the status of order A100?",
     "expected_tools": ["lookup_order"], "expected_answer": ["shipped"]},
    {"task_id": "3", "input": "According to the docs, how many days do customers have to request a refund?",
     "expected_tools": ["search_docs"], "expected_answer": ["30 days", "thirty days"]},
    {"task_id": "4", "input": "Order B200 was delivered some days ago. How many days remain in its refund window?",
     "expected_tools": ["lookup_order", "search_docs", "calculate"], "expected_answer": ["18 days", "eighteen days"]},
    {"task_id": "5", "input": "When should a support agent escalate to a supervisor?",
     "expected_tools": ["search_docs"], "expected_answer": ["twice", "two times", "second time"]},
]


def run_task(agent, task: Dict[str, Any], model: str = MODEL) -> Dict[str, Any]:
    """Run one task and record everything the metrics need."""
    started = time.time()
    result = agent.invoke({"messages": [{"role": "user", "content": task["input"]}]})
    messages = result["messages"]
    tools_used = [call["name"] for m in messages if getattr(m, "tool_calls", None) for call in m.tool_calls]
    usage = [m.usage_metadata for m in messages if getattr(m, "usage_metadata", None)]
    final = messages[-1].content
    answer = final if isinstance(final, str) else json.dumps(final)
    return {
        "task_id": task["task_id"],
        "model": model,
        "answer": answer,
        "tools": tools_used,
        "input_tokens": sum(u["input_tokens"] for u in usage),
        "output_tokens": sum(u["output_tokens"] for u in usage),
        "steps": len(usage),
        "latency_s": round(time.time() - started, 2),
        "success": any(phrase in answer.lower() for phrase in task["expected_answer"]),
        "messages": messages,
    }


# --- Run it ---
agent = build_agent()
runs = [run_task(agent, task) for task in TASKS]

print("=== Evaluation runs on", MODEL, "===")
for run in runs:
    print(f"[{run['task_id']}] success={run['success']!s:5} steps={run['steps']} "
          f"tokens={run['input_tokens']}/{run['output_tokens']} {run['latency_s']:>4}s tools={run['tools']}")
    print(f"     {run['answer'][:110]}")

## Part 2: Task Completion Metrics

The chapter's first listing computes the success rate over a list of task results. The results from Part 1 already carry a `success` flag (the expected substring appeared in the answer) and a step count, so the listing runs as printed.

In [ ]:
from typing import List, Dict

def calculate_success_rate(task_results: List[Dict[str, any]]) -> float:
    """Calculate success rate across agent task evaluations."""
    if not task_results:
        return 0.0

    successful_tasks = sum(1 for result in task_results if result["success"])
    return successful_tasks / len(task_results)


# --- Run it ---
task_results = [{"task_id": r["task_id"], "success": r["success"], "steps": r["steps"]} for r in runs]
success_rate = calculate_success_rate(task_results)
print(f"Success Rate: {success_rate:.2%}")
print(f"Steps per task: {[r['steps'] for r in task_results]} (each step is one model call)")

## Part 3: Reasoning Quality Metrics

Success rate ignores how the agent got there. The chapter's tool selection listing compares the expected tool sequence with the tools the agent actually called, position by position. Because Part 1 recorded the tool calls in order, the metric runs on real traces rather than on the chapter's hand-written example.

In [ ]:
def evaluate_tool_selection(expected_tools: List[str], actual_tools: List[str]) -> float:
    """Evaluate tool selection accuracy."""
    if not expected_tools:
        return 1.0

    correct_selections = sum(
        1 for exp, act in zip(expected_tools, actual_tools) if exp == act
    )
    return correct_selections / len(expected_tools)


# --- Run it ---
print("=== Tool selection accuracy ===")
tool_scores = []
for task, run in zip(TASKS, runs):
    score = evaluate_tool_selection(task["expected_tools"], run["tools"])
    tool_scores.append(score)
    print(f"[{task['task_id']}] expected={task['expected_tools']} actual={run['tools']} -> {score:.0%}")
print(f"\nmean tool selection accuracy: {sum(tool_scores) / len(tool_scores):.2%}")

## Part 4: Cost Efficiency Metrics

### Cost per task and cost per success

The chapter's cost metrics take a `cost_usd` per task. The notebook derives it from the token counts Part 1 recorded and the list price of the model that ran, so the cost per success below is what these five runs actually cost.

In [ ]:
# OpenAI list prices per million tokens, September 2026
PRICES = {
    "gpt-5": (1.25, 10.00),
    "gpt-5-nano": (0.05, 0.40),
    "gpt-5.6-luna": (0.20, 1.20),
    "gpt-5.6-terra": (2.00, 12.00),
}


def cost_usd(model: str, input_tokens: int, output_tokens: int) -> float:
    price_in, price_out = PRICES[model]
    return input_tokens / 1_000_000 * price_in + output_tokens / 1_000_000 * price_out


def calculate_cost_metrics(results: List[Dict]) -> Dict[str, float]:
    """Calculate cost efficiency metrics for agent tasks."""
    total_cost = sum(r["cost_usd"] for r in results)
    total_tasks = len(results)
    successful_tasks = sum(1 for r in results if r["success"])

    avg_cost_per_task = total_cost / total_tasks if total_tasks > 0 else 0
    cost_per_success = (total_cost / successful_tasks
                       if successful_tasks > 0 else float('inf'))

    return {
        "total_cost_usd": total_cost,
        "avg_cost_per_task": avg_cost_per_task,
        "cost_per_success": cost_per_success,
        "success_rate": successful_tasks / total_tasks if total_tasks > 0 else 0
    }


# --- Run it ---
results = [{"task_id": r["task_id"], "success": r["success"],
            "cost_usd": cost_usd(r["model"], r["input_tokens"], r["output_tokens"])} for r in runs]
metrics = calculate_cost_metrics(results)
print(f"Cost per successful task: ${metrics['cost_per_success']:.5f}")
print(f"Total for five tasks:     ${metrics['total_cost_usd']:.5f} at {metrics['success_rate']:.0%} success")
print(f"Mean latency:             {sum(r['latency_s'] for r in runs) / len(runs):.1f}s")

### The CostTracker

The chapter's `CostTracker` records each execution and aggregates cost per success. It ships with GPT-5 and GPT-5 nano rates; the run block adds the two models this notebook uses to the same table before tracking the five runs.

In [ ]:
import time
from typing import Dict

class CostTracker:
    """Track costs for agent executions."""

    # Token costs in USD per million tokens (OpenAI list prices, September 2026)
    COSTS_PER_MILLION_TOKENS = {
        "gpt-5-input": 1.25,
        "gpt-5-output": 10.00,
        "gpt-5-nano-input": 0.05,
        "gpt-5-nano-output": 0.40,
    }

    def __init__(self):
        self.executions = []

    def track_execution(
        self,
        task_id: str,
        model: str,
        input_tokens: int,
        output_tokens: int,
        success: bool
    ):
        """Record execution metrics and costs."""
        input_cost = (input_tokens / 1_000_000) * self.COSTS_PER_MILLION_TOKENS[f"{model}-input"]
        output_cost = (output_tokens / 1_000_000) * self.COSTS_PER_MILLION_TOKENS[f"{model}-output"]
        total_cost = input_cost + output_cost

        self.executions.append({
            "task_id": task_id,
            "model": model,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_cost_usd": total_cost,
            "success": success,
            "timestamp": time.time()
        })

    def get_cost_metrics(self) -> Dict:
        """Calculate aggregate cost metrics."""
        total_cost = sum(e["total_cost_usd"] for e in self.executions)
        successful_tasks = sum(1 for e in self.executions if e["success"])

        return {
            "total_cost_usd": total_cost,
            "total_tasks": len(self.executions),
            "successful_tasks": successful_tasks,
            "cost_per_success": (total_cost / successful_tasks
                               if successful_tasks > 0 else float('inf'))
        }


# --- Run it ---
# The models this notebook calls, at the same September 2026 list prices
CostTracker.COSTS_PER_MILLION_TOKENS.update({
    "gpt-5.6-luna-input": 0.20, "gpt-5.6-luna-output": 1.20,
    "gpt-5.6-terra-input": 2.00, "gpt-5.6-terra-output": 12.00,
})

tracker = CostTracker()
for run in runs:
    tracker.track_execution(run["task_id"], run["model"], run["input_tokens"], run["output_tokens"], run["success"])
print(json.dumps(tracker.get_cost_metrics(), indent=2))

### Model cascading

The chapter's cascading metric asks how much a router saves by sending simple tasks to a cheap model and hard ones to a premium model. The router below is deliberately simple: a task that needs more than one tool goes to `gpt-5.6-terra`, everything else stays on `gpt-5.6-luna`. The cell runs the routed tasks for real, then compares three policies at the recorded token counts: everything on luna, everything on terra, and the router's split. The saving depends on the split and on the price gap, which is 10 to 1 between these two models.

In [ ]:
def route(task: Dict[str, Any]) -> str:
    """Send multi-tool tasks to the premium model, the rest to the small one."""
    return "gpt-5.6-terra" if len(task["expected_tools"]) > 1 else "gpt-5.6-luna"


# --- Run it ---
premium_agent = build_agent("gpt-5.6-terra")
routed_runs = []
for task, luna_run in zip(TASKS, runs):
    model = route(task)
    routed_runs.append(run_task(premium_agent, task, model) if model == "gpt-5.6-terra" else luna_run)

def policy_cost(policy_runs, model=None):
    return sum(cost_usd(model or r["model"], r["input_tokens"], r["output_tokens"]) for r in policy_runs)

all_luna = policy_cost(runs, "gpt-5.6-luna")
all_terra = policy_cost(runs, "gpt-5.6-terra")
routed = policy_cost(routed_runs)

print("=== Model cascading ===")
for task, run in zip(TASKS, routed_runs):
    print(f"[{task['task_id']}] {run['model']:14s} success={run['success']!s:5} tools={run['tools']}")
print(f"\nall luna:   ${all_luna:.5f}")
print(f"all terra:  ${all_terra:.5f}")
print(f"routed:     ${routed:.5f}  ({1 - routed / all_terra:.0%} below all terra, {routed / all_luna:.1f}x all luna)")
print(f"routed success rate: {calculate_success_rate(routed_runs):.0%}")

## Part 5: Benchmarks

The chapter's benchmark section describes AgentBench, GAIA, and SWE-bench rather than listing code, because each ships its own harness and leaderboard. The harness in this notebook is the same shape in miniature: a task set with expected outcomes, an agent, and metrics over the runs. Scaling it up means swapping the five tasks for a benchmark's task set and its checker:

- AgentBench runs agents through eight environments with turn limits (https://github.com/THUDM/AgentBench)
- GAIA scores exact-match answers on 466 questions across three levels, with the leaderboard on Hugging Face (https://huggingface.co/gaia-benchmark)
- SWE-bench and SWE-Bench Pro apply the agent's patch and run the repository's tests (https://github.com/SWE-bench/SWE-bench, https://github.com/scaleapi/SWE-bench_Pro-os)

One caution from the chapter applies to any of them: OpenAI stopped reporting SWE-bench Verified in February 2026 after finding flawed tests and training-data contamination, so a benchmark score is only as good as the benchmark's upkeep.

## Part 6: Observability and Tracing

### LangSmith tracing

The chapter's tracing listing sets three environment variables and lets LangChain trace every run. The cell runs it only when `LANGSMITH_API_KEY` is set, so the notebook does not fail for readers without a LangSmith account. With a key, one traced run lands in the `agent-evaluation` project of your workspace.

In [ ]:
if HAS_LANGSMITH:
    # Enable LangSmith tracing
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = "agent-evaluation"

    # Your agent code automatically gets traced
    llm = ChatOpenAI(model=MODEL, reasoning_effort="none")
    traced_agent = create_agent(llm, TOOLS, system_prompt=SYSTEM_PROMPT)

    # Every execution is automatically traced
    result = traced_agent.invoke({"messages": [{"role": "user", "content": "Research agentic AI frameworks"}]})
    print("traced run complete; open the agent-evaluation project in LangSmith to inspect it")
    print(result["messages"][-1].content[:200])

    os.environ["LANGSMITH_TRACING"] = "false"
else:
    print("skipped: set LANGSMITH_API_KEY in .env to trace runs in LangSmith")

### A trace without a platform

A trace is the ordered record of what the agent did. LangChain keeps that record in the message list a run returns, so the chapter's debugging strategies (trace correlation, comparative analysis, bottleneck identification) can start without any platform. The cell prints task 4's run step by step, which is the same sequence a LangSmith trace would show.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

run = next(r for r in runs if r["task_id"] == "4")
print("=== Trace of task 4 ===")
for message in run["messages"]:
    if isinstance(message, HumanMessage):
        print(f"user      | {message.content}")
    elif isinstance(message, AIMessage) and message.tool_calls:
        for call in message.tool_calls:
            print(f"tool call | {call['name']}({json.dumps(call['args'])})")
    elif isinstance(message, ToolMessage):
        print(f"tool      | {str(message.content)[:80]}")
    elif isinstance(message, AIMessage):
        print(f"agent     | {message.content[:110]}")
print(f"\n{run['steps']} model calls, {run['input_tokens'] + run['output_tokens']} tokens, {run['latency_s']}s")

## Part 7: Production Evaluation Strategies

### A dataset from production traces

The chapter's listing samples successful runs from a LangSmith project and turns them into an evaluation dataset. It needs a LangSmith account, so the cell runs it only when a key is present, against the `agent-evaluation` project that Part 6 populated. Without a key, the cell applies the same production-sampling idea to the local runs and writes the dataset to a file in the workspace, which is what an evaluation job would read.

In [ ]:
from langsmith import Client
from typing import Dict, Optional


def create_eval_dataset_from_production(
    project_name: str,
    sample_size: int,
    run_filter: Optional[str] = None,
) -> str:
    """Create evaluation dataset from production traces."""
    client = Client()

    # Fetch successful root runs; the filter uses LangSmith's query syntax
    runs = client.list_runs(
        project_name=project_name,
        is_root=True,
        filter=run_filter or 'eq(status, "success")',
        limit=sample_size,
    )

    # Extract inputs and outputs, keeping the trace reference as metadata
    examples = [
        {
            "inputs": run.inputs,
            "outputs": run.outputs,
            "metadata": {
                "trace_id": str(run.id),
                "timestamp": run.start_time.isoformat(),
                "tags": run.tags,
            },
        }
        for run in runs
    ]

    # Create dataset
    dataset = client.create_dataset(
        dataset_name=f"{project_name}-eval-{len(examples)}",
        description="Production-sampled evaluation dataset",
    )
    client.create_examples(dataset_id=dataset.id, examples=examples)

    return dataset.name


# --- Run it ---
if HAS_LANGSMITH:
    print("created LangSmith dataset:", create_eval_dataset_from_production("agent-evaluation", sample_size=10))
else:
    print("skipped LangSmith; sampling the local runs instead")

# The same idea offline: successful runs become examples with their inputs, outputs, and provenance
local_examples = [
    {"inputs": {"question": task["input"]}, "outputs": {"answer": run["answer"]},
     "metadata": {"model": run["model"], "tools": run["tools"], "latency_s": run["latency_s"]}}
    for task, run in zip(TASKS, runs) if run["success"]
]
dataset_path = WORKSPACE / "support-agent-eval.jsonl"
dataset_path.write_text("\n".join(json.dumps(ex) for ex in local_examples) + "\n")
print(f"wrote {len(local_examples)} examples to {dataset_path.name}")

### LLM-as-a-judge

Success by substring says nothing about whether an answer was helpful. The chapter's judge asks a model to score helpfulness on a 0 to 1 scale with a reason, using a structured output so the score is a number rather than text to parse. The cell scores all five answers and one deliberately poor answer.

In [ ]:
from typing import Dict
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI


class Judgment(BaseModel):
    score: float = Field(description="Helpfulness from 0.0 to 1.0")
    reasoning: str = Field(description="Two or three sentences justifying the score")


JUDGE = ChatOpenAI(model="gpt-5.6-luna", reasoning_effort="none").with_structured_output(Judgment)


def evaluate_response_quality(agent_output: str, task_description: str) -> Dict:
    """Evaluate agent response quality using LLM-as-judge."""
    prompt = (
        "You are grading an AI agent's response for helpfulness on a 0 to 1 scale. "
        "A helpful response answers the task completely, accurately, and concisely.\n\n"
        f"Task: {task_description}\n\nResponse: {agent_output}"
    )
    result = JUDGE.invoke(prompt)
    return {
        "score": result.score,  # 0-1 scale
        "reasoning": result.reasoning,
        "passed": result.score >= 0.7,
    }


# --- Run it ---
print("=== Judge scores ===")
judgments = {}
for task, run in zip(TASKS, runs):
    judgments[task["task_id"]] = evaluate_response_quality(run["answer"], task["input"])
    j = judgments[task["task_id"]]
    print(f"[{task['task_id']}] score={j['score']:.2f} passed={j['passed']!s:5} | {j['reasoning'][:90]}")

# A deliberately unhelpful answer, as in the chapter's example
task = "Explain how to deploy agents to Kubernetes"
agent_response = "To deploy agents to Kubernetes, create Deployment..."
evaluation = evaluate_response_quality(agent_response, task)
print(f"\nfragment: score={evaluation['score']:.2f} passed={evaluation['passed']}")
print(f"Reasoning: {evaluation['reasoning'][:160]}")

### Calibrating the judge

The chapter says to validate a judge against human judgments and to look for agreement above 80 percent. The dictionary below holds hand-written human scores for the five answers, assigned by reading them. Agreement is measured two ways: the share of tasks where judge and human agree on pass or fail at the 0.7 threshold, and the mean absolute difference between the scores.

In [ ]:
# Human scores, written by reading the five answers in Part 1
HUMAN_SCORES = {"1": 1.0, "2": 0.9, "3": 1.0, "4": 0.9, "5": 0.9}

agree = sum((judgments[t]["score"] >= 0.7) == (HUMAN_SCORES[t] >= 0.7) for t in HUMAN_SCORES)
mean_abs_diff = sum(abs(judgments[t]["score"] - HUMAN_SCORES[t]) for t in HUMAN_SCORES) / len(HUMAN_SCORES)

print("=== Judge calibration ===")
for t in HUMAN_SCORES:
    print(f"[{t}] judge={judgments[t]['score']:.2f} human={HUMAN_SCORES[t]:.2f}")
print(f"\npass/fail agreement: {agree / len(HUMAN_SCORES):.0%} (chapter threshold: 80%)")
print(f"mean absolute score difference: {mean_abs_diff:.2f}")

### Giving the judge the evidence

Read the reasoning the judge gave for tasks 3 and 4. It marked correct answers down because the question alone does not say what the refund window is or when the order was delivered; that information came from the tools. A judge that sees only the question and the answer cannot verify facts the agent looked up. The fix is to pass the tool observations along with the task, so the judge grades the answer against the evidence the agent gathered. The chapter's function takes a task description, so the description now carries the evidence. The cell re-scores the five answers and re-measures agreement with the same human scores.

In [ ]:
def evidence_for(run: Dict[str, Any]) -> str:
    """Collect the tool observations from a run as plain text."""
    observations = [f"{m.name}: {m.content}" for m in run["messages"] if isinstance(m, ToolMessage)]
    return "\n".join(observations) if observations else "(no tools were called)"


# --- Run it ---
print("=== Judge scores with evidence ===")
judgments_with_evidence = {}
for task, run in zip(TASKS, runs):
    description = f"{task['input']}\n\nEvidence the agent gathered with its tools:\n{evidence_for(run)}"
    judgments_with_evidence[task["task_id"]] = evaluate_response_quality(run["answer"], description)
    j = judgments_with_evidence[task["task_id"]]
    print(f"[{task['task_id']}] score={j['score']:.2f} (was {judgments[task['task_id']]['score']:.2f}) | {j['reasoning'][:80]}")

agree_with_evidence = sum(
    (judgments_with_evidence[t]["score"] >= 0.7) == (HUMAN_SCORES[t] >= 0.7) for t in HUMAN_SCORES
)
print(f"\npass/fail agreement: {agree_with_evidence / len(HUMAN_SCORES):.0%} (was {agree / len(HUMAN_SCORES):.0%})")

## Part 8: The Evaluation Report with Regression Gates

Offline evaluation earns its keep as a gate: run the suite against a candidate agent, compute the metrics, and fail the build when any metric crosses a threshold. The cell assembles every metric from the notebook into one report and checks it against gates that a team might set for this agent. The thresholds are illustrative; the pattern is what a CI job would run on every change to the prompt, the tools, or the model.

In [ ]:
report = {
    "model": MODEL,
    "tasks": len(runs),
    "success_rate": calculate_success_rate(runs),
    "tool_selection_accuracy": sum(tool_scores) / len(tool_scores),
    "cost_per_success_usd": tracker.get_cost_metrics()["cost_per_success"],
    "mean_latency_s": sum(r["latency_s"] for r in runs) / len(runs),
    "mean_judge_score": sum(j["score"] for j in judgments_with_evidence.values()) / len(judgments_with_evidence),
    "judge_human_agreement": agree_with_evidence / len(HUMAN_SCORES),
}

GATES = {
    "success_rate": ("min", 0.80),
    "tool_selection_accuracy": ("min", 0.80),
    "cost_per_success_usd": ("max", 0.01),
    "mean_latency_s": ("max", 15.0),
    "mean_judge_score": ("min", 0.70),
    "judge_human_agreement": ("min", 0.80),
}

print("=== Evaluation report ===")
failures = []
for metric, value in report.items():
    if metric in GATES:
        kind, threshold = GATES[metric]
        ok = value >= threshold if kind == "min" else value <= threshold
        if not ok:
            failures.append(metric)
        print(f"{metric:26s} {value:>9.4f}   gate {kind} {threshold:<7} {'pass' if ok else 'FAIL'}")
    else:
        print(f"{metric:26s} {value!s:>9}")

(WORKSPACE / "evaluation-report.json").write_text(json.dumps(report, indent=2))
print("\nall gates passed" if not failures else f"\ngates failed: {failures}")

## Summary

In this notebook, we implemented:

1. **An Agent Under Evaluation**: A tool-using support agent and a five-task evaluation set with expected tool sequences and answers
2. **Task Completion Metrics**: The chapter's success rate over real runs, with step counts from the model calls
3. **Reasoning Quality Metrics**: Tool selection accuracy computed from the tool calls in each run
4. **Cost Efficiency Metrics**: Cost per task and per success from token usage and list prices, the `CostTracker`, and a model cascading comparison across three routing policies
5. **Tracing**: LangSmith tracing when a key is present, and a step-by-step trace read directly from the agent's messages
6. **Production Datasets**: The chapter's LangSmith sampling listing, guarded, and the same idea applied offline to a JSON Lines file
7. **LLM-as-a-Judge**: Structured helpfulness scores with reasoning, calibrated against human scores before and after giving the judge the tool evidence
8. **An Evaluation Report**: Every metric in one report with regression gates, the shape of an offline evaluation in CI

### Key Takeaways:

- Every metric in the chapter needs the same raw material: the final answer, the tool calls in order, token counts, and latency, all of which a LangChain run already exposes
- Success rate and tool selection accuracy answer different questions, and a task can pass one while failing the other
- Cost per success, not cost per task, is the number to compare across models, and routing changes it more than any prompt edit
- A judge is only trustworthy after it has been checked against human scores on the same answers, and it needs the same evidence the agent had
- An evaluation suite becomes useful the day it is wired to a gate that can fail a change

### Next Steps:

- Replace the five tasks with a sample of real user requests and grow the human-scored set until judge agreement is stable
- Run the report cell on every change to the prompt, tools, or model, and keep the reports so regressions show up as a trend
- Move on to Chapter 11, which covers responsible AI, safety, and ethics for agent systems